# MODIS Aqua **nighttime SST** → DBO + Aleutians, EASE-Grid 2.0 North

Companion to `02_modis-chlor-a-daily-ease2.ipynb`, but for **sea-surface temperature** instead of
chlorophyll. Same Pacific-Arctic domain as the PACE NRT map (a separate PACE near-real-time notebook, not included):
the **Aleutian arc / Bering shelf north to the Beaufort (DBO1–8)**, lon −180…−124°E, lat 50…74°N,
gridded onto **EASE-Grid 2.0 North (EPSG:6931)** at 4 km.

- **Product:** `MODIS_AQUA_L3_SST_THERMAL_DAILY_4KM_NIGHTTIME_V2019.0` (PO.DAAC), variable `sst` (°C).
- **Record:** 2002-07-04 → today.
- **Pipeline:** stream each daily L3M file, subset to the region, quality-mask, bucket-average onto
  the EASE-2 grid, write one Zarr per day. **Downloads run in parallel** (8 workers) — the main
  speedup over the sequential chl notebook — and skip-existing lets you stop/resume freely.
- **Analyses (core set):** region SST map, per-box weekly + anomaly trend plots (OLS + Mann-Kendall,
  °C/decade), faceted small-multiples, and a seasonal calendar heatmap.

> ⚠️ **Nighttime-SST caveat:** MODIS nighttime retrievals require the sun below the horizon, so
> during **Arctic polar day (≈ May–Jul above ~66°N)** there are *no* nighttime SST pixels. That is
> exactly the ice-free window for the far-north boxes (DBO4–8), so their summer coverage will be
> thin. The Aleutian/Bering boxes (DBO1–3) have night year-round and are unaffected.

## 1. Imports

In [1]:
import pickle, time as _time
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import earthaccess

from pyresample import AreaDefinition
from pyresample.bucket import BucketResampler
from matplotlib.path import Path as MplPath
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from scipy.stats import kendalltau, norm

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
print("imports ok")

imports ok


## 2. Configuration — product, domain, EASE-2 grid, DBO boxes

Same EASE-Grid 2.0 North subset (EPSG:6931, 4 km, 906×845) as the PACE NRT map, so SST and chl
maps are directly comparable. `DBO_BOXES` are the canonical 4-corner polygons from
`DBO1_8_StationAndBoxCoordinates_DBO4nOnly_072522.xlsx` (sheet `DBOboxes1_8_050522`).

In [ ]:
SST_SHORT_NAME = "MODIS_AQUA_L3_SST_THERMAL_DAILY_4KM_NIGHTTIME_V2019.0"

START_DATE = date(2002, 7, 4)     # MODIS Aqua first light
END_DATE   = date.today()

# Region (matches PACE NRT map): lon_min, lat_min, lon_max, lat_max
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = -180.0, 50.0, -124.0, 74.0

# SST quality + physical range filters
SST_VALID_MIN, SST_VALID_MAX = -2.0, 40.0   # deg C; ocean never below ~-1.8
MAX_QUAL = 1                                 # qual_sst: 0=best, 1=good (keep <=1)

OUT_DIR = Path.home()  / "modis" / "outputs_MODIS_SST"  # add your path
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- EASE-Grid 2.0 North subset, 4 km (identical to the PACE NRT map) ---
dbo_area = AreaDefinition(
    area_id="ease2_north_dbo_4km",
    description="EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km",
    proj_id="EPSG:6931",
    projection={"proj": "laea", "lat_0": 90, "lon_0": 0,
                "x_0": 0, "y_0": 0, "datum": "WGS84", "units": "m"},
    width=906, height=845,
    area_extent=(-3_624_000, 992_000, 0, 4_372_000),
)
print(dbo_area)

# Canonical DBO box polygons, 4 corners each (lat, lon)
DBO_BOXES = {
    "DBO1": [(63.769462, -172.312422), (61.945988, -172.186927), (61.846516, -175.790978), (63.662718, -176.147111)],
    "DBO2": [(65.107747, -170.865176), (65.097001, -167.859549), (64.481828, -167.907758), (64.493658, -170.840669)],
    "DBO3": [(66.785880, -171.330008), (68.608869, -171.419334), (68.572412, -166.481029), (66.752281, -166.755808)],
    "DBO4": [(71.994000, -164.226000), (71.801000, -159.729000), (70.641000, -160.305000), (70.821000, -164.556000)],
    "DBO5": [(71.807730, -158.537691), (71.635583, -155.931220), (71.110941, -156.307277), (71.278277, -158.847920)],
    "DBO6": [(71.850000, -150.108000), (70.890000, -150.976000), (71.159000, -153.865000), (72.119000, -152.980000)],
    "DBO7": [(70.868000, -141.528000), (69.946000, -142.679000), (70.333000, -145.439000), (71.253000, -144.256000)],
    "DBO8": [(70.834000, -125.155000), (70.049000, -126.997000), (70.664000, -129.354000), (71.442000, -127.431000)],
}
print(f"\nWindow: {START_DATE} -> {END_DATE}   product: {SST_SHORT_NAME}")

Area ID: ease2_north_dbo_4km
Description: EASE-Grid 2.0 North subset: Aleutians -> Beaufort (DBO1-8), 4 km
Projection ID: EPSG:6931
Projection: {'datum': 'WGS84', 'lat_0': '90', 'lon_0': '0', 'no_defs': 'None', 'proj': 'laea', 'type': 'crs', 'units': 'm', 'x_0': '0', 'y_0': '0'}
Number of columns: 906
Number of rows: 845
Area extent: (-3624000, 992000, 0, 4372000)

Window: 2002-07-04 -> 2026-06-15   product: MODIS_AQUA_L3_SST_THERMAL_DAILY_4KM_NIGHTTIME_V2019.0


## 3. EarthData authentication

In [13]:
auth = earthaccess.login(persist=True)
print("authenticated:", earthaccess.__auth__.authenticated)

authenticated: True


## 4. Search the daily granules

Every granule in this collection is already daily / 4 km / nighttime, so we just keep one per
calendar day (preferring the **standard** product over `.NRT`, which covers recent weeks).

> **Robustness note:** CMR's granule endpoint **500-errors intermittently** for this PO.DAAC
> collection, and one large paginated query gets **silently truncated** (a single `.get()` stopped
> at 2011 even though the record runs to today). So we walk the archive **month-by-month with
> retries** — each small window is reliable, and together they capture the full 2002→present record.
> This search takes a few minutes; it only needs to run once to drive the download loop.

In [6]:
import time as _t, calendar

def granule_date(g):
    return date.fromisoformat(g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10])

def is_nrt(g):
    return ".NRT" in g["umm"]["GranuleUR"]

def _fetch(s, e, tries=6):
    """One temporal-window query, retried with backoff. CMR's granule endpoint 500s
    intermittently for this collection, so we retry; small windows also avoid the
    silent truncation that large paginated queries suffer."""
    for k in range(tries):
        try:
            return earthaccess.DataGranules().short_name(SST_SHORT_NAME).temporal(s, e).get(500)
        except Exception:
            _t.sleep(2 * (k + 1))
    raise RuntimeError(f"CMR failed for {s}..{e} after {tries} tries")

# Walk month-by-month (<=~64 granules/window — safely under the page cap and reliable)
all_granules = []
cur = date(START_DATE.year, START_DATE.month, 1)
while cur <= END_DATE:
    last = date(cur.year, cur.month, calendar.monthrange(cur.year, cur.month)[1])
    all_granules.extend(_fetch(max(cur, START_DATE).isoformat(),
                               min(last, END_DATE).isoformat()))
    if cur.month == 1:
        print(f"  ...{cur.year}  (running total {len(all_granules)})", end="\r")
    cur = date(cur.year + cur.month // 12, cur.month % 12 + 1, 1)
print(f"\nFetched {len(all_granules)} granules over {START_DATE} -> {END_DATE}")

granules_by_day = {}
for g in all_granules:
    d = granule_date(g)
    # keep one per day; upgrade NRT -> standard if a standard granule also exists
    if d not in granules_by_day or (is_nrt(granules_by_day[d]) and not is_nrt(g)):
        granules_by_day[d] = g
print(f"Unique days: {len(granules_by_day)}  ({min(granules_by_day)} -> {max(granules_by_day)})")
for g in all_granules[:3]:
    print("  UR:", g["umm"]["GranuleUR"])

  ...2026  (running total 11196)
Fetched 11423 granules over 2002-07-04 -> 2026-06-12
Unique days: 8719  (2002-07-04 -> 2026-06-09)
  UR: AQUA_MODIS.20020704.L3m.DAY.NSST.sst.4km
  UR: AQUA_MODIS.20020705.L3m.DAY.NSST.sst.4km
  UR: AQUA_MODIS.20020706.L3m.DAY.NSST.sst.4km


## 5. Per-day processor — stream, subset, quality-mask, bucket-average → Zarr

MODIS L3M is a global 8640×4320 grid with `lat` descending 90→−90. We slice straight to the
region (`lat slice(74, 50)`, `lon slice(-180, -124)`) so each file reads only ~1300×600 pixels,
then bucket-average onto the EASE-2 grid. One Zarr per day; skip-existing makes re-runs cheap.

In [7]:
def write_zarr_sst(arr_2d, out_path, area, source_granule_id, t):
    xc, yc = area.get_proj_vectors()
    ds = xr.Dataset(
        {"sst": (("y", "x"), arr_2d.astype(np.float32))},
        coords={"x": ("x", xc), "y": ("y", yc), "time": t},
        attrs={"source_granule": source_granule_id, "source_product": SST_SHORT_NAME,
               "target_grid": area.area_id, "projection": area.proj_str,
               "created_by": "06_modis-sst-daily.ipynb"},
    )
    ds["sst"].attrs.update({"units": "degree_C", "long_name": "Sea surface temperature (nighttime)"})
    ds.to_zarr(out_path, mode="w")

def process_one_day(granule, area, out_dir):
    d = granule_date(granule)
    zpath = out_dir / f"a{d.year}{d.timetuple().tm_yday:03d}_sst_MODISA.zarr"
    if zpath.exists():
        return "skip"

    files = earthaccess.open([granule], show_progress=False)
    with xr.open_dataset(files[0], engine="h5netcdf", chunks={}) as ds:
        sub = ds["sst"].sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
        sst = sub.values
        lat1d, lon1d = sub["lat"].values, sub["lon"].values
        qual = (ds["qual_sst"].sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX)).values
                if "qual_sst" in ds.variables else None)

    valid = np.isfinite(sst) & (sst > SST_VALID_MIN) & (sst < SST_VALID_MAX)
    if qual is not None:
        valid &= (qual <= MAX_QUAL)
    if not valid.any():
        write_zarr_sst(np.full((area.height, area.width), np.nan), zpath, area,
                       granule["umm"]["GranuleUR"], np.datetime64(d.isoformat()))
        return "empty"
    sst = np.where(valid, sst, np.nan)

    LON, LAT = np.meshgrid(lon1d, lat1d)
    resampler = BucketResampler(area, da.from_array(LON, chunks="auto"),
                                       da.from_array(LAT, chunks="auto"))
    grid_sst = resampler.get_average(da.from_array(sst, chunks="auto"))
    if hasattr(grid_sst, "compute"):
        grid_sst = grid_sst.compute()
    write_zarr_sst(grid_sst, zpath, area, granule["umm"]["GranuleUR"],
                   np.datetime64(d.isoformat()))
    return "wrote"

print("per-day processor ready")

per-day processor ready


## 6. Parallel daily loop (8 workers) — the speed step

Each day spends most of its time waiting on NASA's S3, so we download/process several concurrently.
Skip-existing means you can interrupt and re-run to resume. For a first pass, set `LIMIT` to a small
number (e.g. process just the last 60 days) to validate end-to-end, then set `LIMIT = None` to run
the whole 2002→now record.

In [10]:
MAX_WORKERS = 8
LIMIT = None      # None = full 2002->now record; set an int to process only the last N days

days = sorted(granules_by_day.items())
if LIMIT:
    days = days[-LIMIT:]
print(f"Processing {len(days)} days with {MAX_WORKERS} workers "
      f"({days[0][0]} -> {days[-1][0]})")

counts = defaultdict(int)
t0 = _time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = {ex.submit(process_one_day, g, dbo_area, OUT_DIR): d for d, g in days}
    for k, fut in enumerate(as_completed(futs), 1):
        d = futs[fut]
        try:
            counts[fut.result()] += 1
        except Exception as e:
            counts["error"] += 1
            print(f"  {d}  ERR {type(e).__name__}: {e}")
        if k % 50 == 0 or k == len(futs):
            print(f"  {k}/{len(futs)}  {dict(counts)}  elapsed {(_time.time()-t0)/60:.1f} min", end="\r")

print(f"\nDone: {dict(counts)}")
print(f"Total SST Zarrs on disk: {len(list(OUT_DIR.glob('a*_sst_MODISA.zarr')))}")

Processing 8719 days with 8 workers (2002-07-04 -> 2026-06-09)


  2019-07-17  ERR FileNotFoundError: https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/MODIS_AQUA_L3_SST_THERMAL_DAILY_4KM_NIGHTTIME_V2019.0/AQUA_MODIS.20190718.L3m.DAY.NSST.sst.4km.nc


  2023-04-24  ERR FSTimeoutError: 


  8719/8719  {'wrote': 8652, 'empty': 5, 'error': 2, 'skip': 60}  elapsed 364.0 min
Done: {'wrote': 8652, 'empty': 5, 'error': 2, 'skip': 60}
Total SST Zarrs on disk: 8717


## 7. Load all daily Zarrs + per-box time series (cached)

Concatenates the per-day Zarrs into `ds_all`, builds a polygon mask per DBO box on the EASE-2 grid,
and precomputes the daily spatial mean/median SST inside each box. The result is pickled to
`box_ts_cache.pkl` so re-runs are instant — delete that file to force a recompute.

In [18]:
zarr_paths = sorted(OUT_DIR.glob("a*_sst_MODISA.zarr"))
if not zarr_paths:
    raise RuntimeError("No SST Zarrs yet — run the daily loop (cell 6) first.")
ds_all = xr.concat([xr.open_zarr(p) for p in zarr_paths], dim="time").sortby("time")
n_days = ds_all.sizes["time"]
print(f"Loaded {n_days} days "
      f"({str(ds_all.time.values[0])[:10]} -> {str(ds_all.time.values[-1])[:10]})")

# Per-box masks on the EASE-2 grid (MplPath wants (lon, lat) order)
lons2d, lats2d = dbo_area.get_lonlats()
grid_pts = np.column_stack([lons2d.ravel(), lats2d.ravel()])
box_masks = {name: MplPath([(lo, la) for la, lo in corners] + [(corners[0][1], corners[0][0])])
                       .contains_points(grid_pts).reshape(lons2d.shape)
             for name, corners in DBO_BOXES.items()}
for name, m in box_masks.items():
    print(f"  {name}: {m.sum():>5,} grid cells")

# Cache keyed on day count -> auto-recompute when more days are ingested. Delete the .pkl to force.
pkl_path = OUT_DIR / "box_ts_cache.pkl"
box_ts = None
if pkl_path.exists():
    cached = pickle.load(open(pkl_path, "rb"))
    if len(next(iter(cached.values()))["daily_mean"]) == n_days:
        box_ts = cached
        print(f"Loaded box_ts from cache ({n_days} days)")
    else:
        print(f"Cache is stale ({len(next(iter(cached.values()))['daily_mean'])} vs {n_days}) - recomputing")

if box_ts is None:
    # FAST: crop to the DBO bounding box and reduce in memory (~1 min) instead of
    # streaming the full ~27 GB archive per box (which takes ~35 min).
    print("Computing per-box time series (crop to DBO region, in-memory)…")
    H, W = dbo_area.height, dbo_area.width
    anym = np.zeros((H, W), bool)
    for _m in box_masks.values():
        anym |= _m
    ys, xs = np.where(anym)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    sub = ds_all["sst"].isel(y=slice(y0, y1), x=slice(x0, x1)).values   # one small read (~6 GB)
    box_ts = {}
    for name, _m in box_masks.items():
        cells = sub[:, _m[y0:y1, x0:x1]]
        box_ts[name] = {
            "daily_mean":   xr.DataArray(np.nanmean(cells, axis=1),   coords={"time": ds_all.time}, dims="time"),
            "daily_median": xr.DataArray(np.nanmedian(cells, axis=1), coords={"time": ds_all.time}, dims="time"),
        }
    pickle.dump(box_ts, open(pkl_path, "wb"))
    print(f"✓ cached -> {pkl_path.name} ({n_days} days)")

Loaded 8717 days (2002-07-04 -> 2026-06-09)
  DBO1: 2,412 grid cells
  DBO2:   605 grid cells
  DBO3: 2,561 grid cells
  DBO4: 1,293 grid cells
  DBO5:   351 grid cells
  DBO6:   739 grid cells
  DBO7:   773 grid cells
  DBO8:   760 grid cells
Loaded box_ts from cache (8717 days)


## 12. Monthly SST per pixel -> `monthly_sst_cache.nc`

This cache is what `fig03_4a_6_S3_S4_re-do-figs.ipynb` (Fig. S3 SST panel) and `fig03_4a_6_S3_S4_re-do-figs.ipynb` pick up; nothing else from the old animation section is needed.

In [ ]:
# monthly SST per pixel — reuse if in memory, else load cache, else compute (one heavy read)
mon_path = OUT_DIR / "monthly_sst_cache.nc"
if "monthly_sst" not in dir():
    if mon_path.exists():
        monthly_sst = xr.open_dataarray(mon_path)
        print("loaded monthly_sst from cache")
    else:
        print("computing monthly SST per pixel (reads full archive, ~few min)…")
        monthly_sst = ds_all["sst"].resample(time="1MS").mean().compute()
        monthly_sst.to_netcdf(mon_path)
        print(f"cached -> {mon_path.name}")
print(f"{monthly_sst.sizes['time']} monthly frames -> {mon_path}")


## 14. Annual growing-season SST per pixel (shared input for the GIF + trend map)

Per-pixel **May–September** mean SST for each year. This is the one heavy step (it reads the full
archive once to make monthly means, ~a few minutes) and is cached to `annual_sst_cache.nc` so the
GIF and trend-map cells below are fast on re-runs.

In [25]:
GROW_MONTHS = [5, 6, 7, 8, 9]          # May–Sep growing season
ann_path = OUT_DIR / "annual_sst_cache.nc"
if ann_path.exists():
    annual_sst = xr.open_dataarray(ann_path)
    print(f"Loaded annual_sst from cache ({ann_path.name})")
else:
    print("Computing monthly -> annual (May–Sep) SST per pixel (reads full archive, ~few min)…")
    monthly_sst = ds_all["sst"].resample(time="1MS").mean().compute()
    grow = monthly_sst.sel(time=monthly_sst["time.month"].isin(GROW_MONTHS))
    annual_sst = grow.groupby("time.year").mean(dim="time", skipna=True)
    annual_sst.to_netcdf(ann_path)
    print(f"cached -> {ann_path.name}")
years = annual_sst["year"].values
print(f"years: {years.min()}–{years.max()} ({len(years)} annual frames)")

Computing monthly -> annual (May–Sep) SST per pixel (reads full archive, ~few min)…
cached -> annual_sst_cache.nc
years: 2002–2026 (25 annual frames)
